**Import extracted data** (dichotomous outcomes)

In [5]:
import pandas as pd
df = pd.read_csv(f"./data/dichotomous.csv", encoding = "utf-8")
globals()[f"dichotomous"] = df
df = df.drop(columns = ["Unnamed: 0"])
df.head()

,id,study,subgroup,outcome,ni,yi,age,male,female,follow_up
0,6,"Anz et al., 2023",BPTB,graft_failure,8,0,29.4,4,6.0,12.0
1,6,"Anz et al., 2023",HT,graft_failure,8,0,27.6,3,7.0,12.0
2,43,"Kautzner et al., 2015",BPTB,graft_failure,74,0,26.0,0,74.0,24.0
3,43,"Kautzner et al., 2015",HT,graft_failure,73,2,26.0,0,73.0,24.0
4,49,"Laxdal et al., 2005",BPTB,graft_failure,40,1,28.0,0,0.0,26.0


In [1]:
outcomes = [
    "pivot_shift_1",
    "pivot_shift_2",
    "lachman_1",
    "lachman_2",
    "graft_failure"
]

for x in outcomes:
    import pandas as pd
    df = dichotomous[dichotomous["outcome"] == x]
    import os
    df.to_csv(f"./data/extracted/csv/{x}.csv", encoding = "utf-8")
    df.to_excel(f"./data/extracted/xlsx/{x}.xlsx")

    globals()[x] = df

In [2]:
pivot_shift_1.head()

,Unnamed: 0,id,study,subgroup,outcome,ni,yi,age,male,female,follow_up
24,3.0,2,"Aglietti et al., 2004",BPTB,pivot_shift_1,60,10,25.0,46,14.0,24.0
25,3.0,2,"Aglietti et al., 2004",HT,pivot_shift_1,60,11,25.0,46,14.0,24.0
26,7.0,5,"Anderson et al., 2001",BPTB,pivot_shift_1,35,18,23.6,23,12.0,34.6
27,7.0,5,"Anderson et al., 2001",HT,pivot_shift_1,70,36,21.0,45,23.0,35.8
28,19.0,11,"Beynnon et al., 2002",BPTB,pivot_shift_1,22,3,28.5,18,10.0,36.0


2. **Pivot the data** (one row per study)

In [5]:
outcomes = [
    "pivot_shift_1",
    "pivot_shift_2",
    "lachman_1",
    "lachman_2",
    "graft_failure"
]

for outcome in outcomes:
    df = pd.read_csv(f"./data/extracted/csv/{outcome}.csv")
    globals()[outcome] = df

In [6]:
pivot_shift_2.head()

,Unnamed: 0.1,Unnamed: 0,id,study,subgroup,outcome,ni,yi,age,male,female,follow_up
0,74,3.0,2,"Aglietti et al., 2004",BPTB,pivot_shift_2,60,0,25.0,46,14.0,24.0
1,75,3.0,2,"Aglietti et al., 2004",HT,pivot_shift_2,60,0,25.0,46,14.0,24.0
2,76,7.0,5,"Anderson et al., 2001",BPTB,pivot_shift_2,35,15,23.6,23,12.0,34.6
3,77,7.0,5,"Anderson et al., 2001",HT,pivot_shift_2,70,27,21.0,45,23.0,35.8
4,78,19.0,11,"Beynnon et al., 2002",BPTB,pivot_shift_2,22,6,28.5,18,10.0,36.0


In [7]:
lachman_1.head().fillna("")

,Unnamed: 0.1,Unnamed: 0,id,study,subgroup,outcome,ni,yi,age,male,female,follow_up
0,122,19.0,11,"Beynnon et al., 2002",BPTB,lachman_1,22,12,28.5,18,10.0,36.0
1,123,19.0,11,"Beynnon et al., 2002",HT,lachman_1,22,19,29.9,13,15.0,36.0
2,124,,13,"BjÃ¶rnsson et al., 2016",BPTB,lachman_1,61,31,,,,
3,125,,13,"BjÃ¶rnsson et al., 2016",HT,lachman_1,86,67,,,,
4,126,31.0,23,"Ejerhed et al., 2003",BPTB,lachman_1,32,14,27.3,21,11.0,24.0


In [8]:
dfs = {
    "pivot_shift_1": pivot_shift_1,
    "pivot_shift_2": pivot_shift_2,
    "lachman_1": lachman_1,
    "lachman_2": lachman_2,
    "graft_failure": graft_failure
}


for name, df in dfs.items():
    df["group"] = df.groupby(["study"]).cumcount() + 1
    
    # reshape
    wide = (
        df.pivot(
            index= "study",
            columns="group",
            values=["subgroup", "ni", "yi", "age", "male", "female", "follow_up"]
        )
    )
    
    # flatten column names
    wide.columns = [
        f"{name}{group}" if name != "subgroup" else f"treat{group}"
        for name, group in wide.columns
    ]

    wide = (wide.rename(columns = {
     #   "study1": "study",
        "ni1": "n1",
        "ni2": "n2",
        "yi1": "y1",
        "yi2": "y2"
    })).reset_index().fillna("")

    
  #  wide.drop(columns = ["study2"])
    wide.to_csv(f"./data/reformatted/csv/{name}.csv", encoding = "utf-8")
    
    globals()[name] = wide

3. **Prepare sheets for analysis** (calculate odds ratios, standard errors of odds ratios, and log of odds ratios (double check if TE should be log of OR or just OR) as TE, seTE, and logTE respectively)\
`./data/sheets_for_analysis`

In [12]:
outcomes = [
    "pivot_shift_1",
    "pivot_shift_2",
    "lachman_1",
    "lachman_2",
    "graft_failure"
]

for outcome in outcomes:
    import pandas as pd
    import numpy as np
    df = pd.read_csv(f"./data/reformatted/csv/{outcome}.csv", encoding = "utf-8")
    var = ["y1", "y2", "n1", "n2"]
    for v in var:
        df[f"{v}"] = df[f"{v}"] + 0.5
    df["TE"] = ((df["y1"]*(df["n2"] - df["y2"]))/(df["y2"]*(df["n1"] - df["y1"]))).round(3).fillna("")
    df["seTE"] = ((1/df["y1"]) + (1/(df["n1"] - df["y1"])) + (1/df["y2"]) + (1/(df["n2"] - df["y2"])))**0.5
    df["seTE"] = df["seTE"].round(3)
    df.to_csv(f"./data/sheets_for_analysis/csv/{outcome}.csv", encoding = "utf-8")
    df.to_excel(f"./data/sheets_for_analysis/xlsx/{outcome}.xlsx")
    globals()[outcome] = df

In [13]:
pivot_shift_2.head()

,Unnamed: 0,study,treat1,treat2,n1,n2,y1,y2,age1,age2,male1,male2,female1,female2,follow_up1,follow_up2,TE,seTE
0,0,"Aglietti et al., 2004",BPTB,HT,60.5,60.5,0.5,0.5,25.0,25.0,46,46.0,14.0,14.0,24.0,24.0,1.000,2.008
1,1,"Anderson et al., 2001",BPTB,HT,35.5,70.5,15.5,27.5,23.6,21.0,23,45.0,12.0,23.0,34.6,35.8,1.212,0.417
2,2,"Beynnon et al., 2002",BPTB,HT,22.5,22.5,6.5,1.5,28.5,29.9,18,13.0,10.0,15.0,36.0,36.0,5.688,0.965
3,3,"Bi et al., 2018",HT,PLT,62.5,62.5,1.5,0.5,27.9,29.1,31,34.0,31.0,28.0,30.3,29.7,3.049,1.643
4,4,"Björnsson et al., 2016",BPTB,HT,60.5,86.5,12.5,13.5,28.2,26.8,42,53.0,19.0,33.0,202.6,191.9,1.408,0.434
